In [1]:
from pathlib import Path
import torch
from torchvision.models import vgg16

device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

# Load the pretrained VGG-16 weights.
model = vgg16(weights=None)
weights_path = Path("checkpoints/pretrained_vgg.pt")
state_dict = torch.load(
    weights_path, map_location="cpu", weights_only=True
)
model.load_state_dict(state_dict)

# Keep only the convolutional layers and freeze them.
encoder = model.features
encoder.requires_grad_(False)
encoder.eval()
encoder = encoder.to(device)

del model, state_dict

# Check the output dimensions with a dummy image.
with torch.inference_mode():
    features = encoder(torch.zeros(1, 3, 224, 224, device=device))
    image_vector = features.mean(dim=(2, 3))

print("Device:", device)
print("Spatial features:", tuple(features.shape))
print("Image vector:", tuple(image_vector.shape))


Device: mps
Spatial features: (1, 512, 7, 7)
Image vector: (1, 512)
